In [1]:
import pandas as pd

CSV_FILE_NAME = "retell_calls_mindflow.csv"

# 1. FIX DE NOTAÇÃO CIENTÍFICA:
# Forçamos o Pandas a ler o telefone e IDs como TEXTO puro (String) para não perder nenhum dígito.
tipos = {
    'to_number': str,
    'from_number': str,
    'call_id': str
}

df_original = pd.read_csv(CSV_FILE_NAME, dtype=tipos, low_memory=False)

# 2. FIX DA DUPLICAÇÃO FANTASMA:
# Vamos garantir que não existam logs duplicados para a mesma ligação.
tamanho_antes = len(df_original)

# Ordenamos por data e mantemos apenas o primeiro log de cada call_id
df_original = df_original.sort_values('created_at').drop_duplicates(subset=['call_id'], keep='first')

tamanho_depois = len(df_original)
print(f"Ingestão concluída. Linhas originais: {tamanho_antes} | Após remover duplicatas (logs fantasmas): {tamanho_depois}")
print(f"Total de duplicatas removidas: {tamanho_antes - tamanho_depois}")

Ingestão concluída. Linhas originais: 150423 | Após remover duplicatas (logs fantasmas): 45676
Total de duplicatas removidas: 104747


In [2]:
len(df_original)

45676

In [3]:
import pandas as pd

# Converter a coluna created_at para datetime e ajustar para o fuso horário de Brasília
df_original['created_at'] = pd.to_datetime(df_original['created_at']).dt.tz_convert('America/Sao_Paulo')

# Exibir as primeiras linhas para conferir o novo horário
display(df_original[['id', 'created_at', 'Numero']].head())

,id,created_at,Numero
0,5630,2025-12-01 10:59:39.479625-03:00,+5542999425406
187,5817,2025-12-11 10:13:09.239712-03:00,+5511964014443
192,5822,2025-12-11 14:06:13.089003-03:00,+5548988061638
195,5824,2025-12-11 16:49:35.157291-03:00,+5519998624806
198,5827,2025-12-11 16:49:35.951506-03:00,+5511014811851


In [4]:
# 1. Definir o limite de tempo em milissegundos (1.5 minutos = 90.000 ms)
LIMITE_MS = 90000

# 2. Garantir que a coluna 'Duracao' seja numérica
df_original['Duracao_num'] = pd.to_numeric(df_original['Duracao'], errors='coerce').fillna(0)

# 3. Identificar quais 'to_number' tiveram pelo menos uma chamada > 90.000ms
numeros_com_sucesso = df_original[df_original['Duracao_num'] > LIMITE_MS]['to_number'].unique()

# 4. Criar a coluna target: 1 se o número está na lista de sucessos, 0 caso contrário
df_original['target'] = df_original['to_number'].isin(numeros_com_sucesso).astype(int)

# Exibir resultados atualizados
print("Distribuição da coluna Target (Baseada em 90.000ms na coluna Duracao):")
print(df_original['target'].value_counts())

# Mostrar exemplo para conferência
display(df_original[['to_number', 'Duracao', 'target']].head(10))

Distribuição da coluna Target (Baseada em 90.000ms na coluna Duracao):
target
0    44122
1     1554
Name: count, dtype: int64


,to_number,Duracao,target
0,NaN,NaN,0
187,+5511964014443,NaN,0
192,+5548988061638,NaN,0
195,+5519998624806,NaN,0
198,+5511014811851,NaN,0
199,+5521991121433,NaN,0
201,+5518996510939,NaN,0
203,+5511980669905,NaN,0
205,+5548988498926,NaN,0
207,+5541999128101,NaN,0


In [5]:
import numpy as np

# Garantir que os dados estejam ordenados por número e data para os cálculos cumulativos
df_original = df_original.sort_values(['to_number', 'created_at'])

# 1. n_tentativas_anteriores
# Contamos a posição da chamada atual para o número e subtraímos 1 (pois a primeira tentativa deve ser 0)
df_original['n_tentativas_anteriores'] = df_original.groupby('to_number').cumcount()

# 2. horas_desde_primeiro_contato
# Pegamos a data da primeira chamada de cada número
df_original['primeiro_contato'] = df_original.groupby('to_number')['created_at'].transform('min')

# Calculamos a diferença em horas
diff_tempo = df_original['created_at'] - df_original['primeiro_contato']
df_original['horas_desde_primeiro_contato'] = diff_tempo.dt.total_seconds() / 3600

# Limpeza: remover coluna auxiliar e exibir resultados
df_original = df_original.drop(columns=['primeiro_contato'])

print("Novas colunas de métricas de tentativas criadas com sucesso!")
display(df_original[['to_number', 'created_at', 'n_tentativas_anteriores', 'horas_desde_primeiro_contato']].head(15))

Novas colunas de métricas de tentativas criadas com sucesso!


,to_number,created_at,n_tentativas_anteriores,horas_desde_primeiro_contato
115958,+13213177885,2025-12-19 12:00:00.767760-03:00,0.0,0.000000
80958,+13213177885,2025-12-19 12:06:22.369088-03:00,1.0,0.106000
1669,+18589523698,2025-12-19 15:14:25.037069-03:00,0.0,0.000000
133528,+33611677762,2025-12-19 12:50:20.725166-03:00,0.0,0.000000
1625,+33611677762,2025-12-19 12:55:32.230075-03:00,1.0,0.086529
133956,+351962901865,2025-12-18 16:01:18.579516-03:00,0.0,0.000000
133974,+351962901865,2025-12-18 16:06:28.478176-03:00,1.0,0.086083
115656,+351962901865,2025-12-19 12:20:28.533663-03:00,2.0,20.319432
1662,+351962901865,2025-12-19 12:25:38.242083-03:00,3.0,20.405462
39327,+48996112406,2026-03-26 15:04:32.690145-03:00,0.0,0.000000


In [6]:
# 1. Extrair o DDD da coluna 'to_number'
# O DDD no Brasil são os dígitos 3 e 4 (ex: +5511... o DDD é 11)
df_original['ddd'] = df_original['to_number'].str.extract(r'\+55(\d{2})')

# 2. Mapeamento de DDD para Estado (Região)
ddd_to_state = {
    '11': 'SP', '12': 'SP', '13': 'SP', '14': 'SP', '15': 'SP', '16': 'SP', '17': 'SP', '18': 'SP', '19': 'SP',
    '21': 'RJ', '22': 'RJ', '24': 'RJ', '27': 'ES', '28': 'ES', '31': 'MG', '32': 'MG', '33': 'MG', '34': 'MG',
    '35': 'MG', '37': 'MG', '38': 'MG', '41': 'PR', '42': 'PR', '43': 'PR', '44': 'PR', '45': 'PR', '46': 'PR',
    '47': 'SC', '48': 'SC', '49': 'SC', '51': 'RS', '53': 'RS', '54': 'RS', '55': 'RS', '61': 'DF', '62': 'GO',
    '63': 'TO', '64': 'GO', '65': 'MT', '66': 'MT', '67': 'MS', '68': 'AC', '69': 'RO', '71': 'BA', '73': 'BA',
    '74': 'BA', '75': 'BA', '77': 'BA', '79': 'SE', '81': 'PE', '82': 'AL', '83': 'PB', '84': 'RN', '85': 'CE',
    '86': 'PI', '87': 'PE', '88': 'CE', '89': 'PI', '91': 'PA', '92': 'AM', '93': 'PA', '94': 'PA', '95': 'RR',
    '96': 'AP', '97': 'AM', '98': 'MA', '99': 'MA'
}

df_original['Regiao'] = df_original['ddd'].map(ddd_to_state)

# Exibir contagem por Estado para verificar se a extração funcionou
print("Distribuição por Estado (Regiao):")
display(df_original['Regiao'].value_counts())

# Mostrar exemplo das novas colunas
display(df_original[['to_number', 'ddd', 'Regiao']].head())

Distribuição por Estado (Regiao):


,count
Regiao,
SP,18590
SC,7305
RJ,4327
PR,2812
MG,2700
RS,1976
BA,1309
DF,1080
ES,995


,to_number,ddd,Regiao
115958,+13213177885,NaN,NaN
80958,+13213177885,NaN,NaN
1669,+18589523698,NaN,NaN
133528,+33611677762,NaN,NaN
1625,+33611677762,NaN,NaN


In [11]:
import pandas as pd

# 1. Lista de motivos para monitorar
reasons = [
    'voicemail_reached', 'dial_no_answer', 'inactivity', 'invalid_destination',
    'user_hangup', 'user_declined', 'telephony_provider_permission_denied',
    'dial_busy', 'telephony_provider_unavailable', 'agent_hangup', 'error_asr',
    'error_retell', 'dial_failed', 'max_duration_reached', 'ivr_reached'
]

# 2. Garantir ordenação cronológica por número para o cálculo de série temporal
df_original = df_original.sort_values(['to_number', 'created_at'])

# 3. Calcular somas acumulativas anteriores para cada motivo (Evitando Data Leakage)
for reason in reasons:
    # Definir o nome da coluna conforme solicitado (lowercase/TitleCase)
    if reason in ['voicemail_reached', 'dial_no_answer', 'inactivity']:
        col_name = f'n_{reason}_anteriores'
    else:
        col_name = f'N_{reason}_anteriores'

    # Criar uma flag binária para o motivo atual
    is_reason = (df_original['disconnection_reason'] == reason).astype(int)

    # Soma acumulada por grupo de telefone, deslocando 1 posição (shift) para pegar apenas o PASSADO
    df_original[col_name] = df_original.groupby('to_number')[is_reason.name].transform(lambda x: is_reason.loc[x.index].cumsum().shift(1)).fillna(0)

# 4. Adicionar o Status da Última Interação (shift de 1)
df_original['ultima_disconnection_reason'] = df_original.groupby('to_number')['disconnection_reason'].shift(1).fillna('primeiro_contato')

print("Colunas de histórico de desconexão e última interação criadas com sucesso!")

# Exibir exemplo para conferência (pegando um número com mais de 3 tentativas)
try:
    numero_exemplo = df_original[df_original['n_tentativas_anteriores'] > 3]['to_number'].iloc[0]
    colunas_foco = ['to_number', 'created_at', 'disconnection_reason', 'ultima_disconnection_reason', 'n_voicemail_reached_anteriores', 'N_user_hangup_anteriores']
    display(df_original[df_original['to_number'] == numero_exemplo][colunas_foco].head(10))
except:
    display(df_original[['to_number', 'ultima_disconnection_reason']].head())


Colunas de histórico de desconexão e última interação criadas com sucesso!


,to_number,created_at,disconnection_reason,ultima_disconnection_reason,n_voicemail_reached_anteriores,N_user_hangup_anteriores
39327,+48996112406,2026-03-26 15:04:32.690145-03:00,telephony_provider_permission_denied,primeiro_contato,0.0,0.0
39362,+48996112406,2026-03-26 15:09:38.238406-03:00,telephony_provider_permission_denied,telephony_provider_permission_denied,0.0,0.0
39396,+48996112406,2026-03-26 15:14:43.474294-03:00,telephony_provider_permission_denied,telephony_provider_permission_denied,0.0,0.0
39482,+48996112406,2026-03-26 15:29:30.430273-03:00,telephony_provider_permission_denied,telephony_provider_permission_denied,0.0,0.0
39487,+48996112406,2026-03-26 15:30:11.024943-03:00,telephony_provider_permission_denied,telephony_provider_permission_denied,0.0,0.0
77007,+48996112406,2026-03-26 15:30:49.213540-03:00,telephony_provider_permission_denied,telephony_provider_permission_denied,0.0,0.0
140864,+48996112406,2026-03-27 11:28:37.027535-03:00,telephony_provider_permission_denied,telephony_provider_permission_denied,0.0,0.0
140918,+48996112406,2026-03-27 11:33:44.001496-03:00,telephony_provider_permission_denied,telephony_provider_permission_denied,0.0,0.0
77051,+48996112406,2026-03-27 16:47:37.247295-03:00,telephony_provider_permission_denied,telephony_provider_permission_denied,0.0,0.0


In [13]:
df_original.describe()

,id,agent_version,combined_cost,LLM_token_usage,Duracao,segmento,equipe,Duracao_num,target,n_tentativas_anteriores,...,N_user_declined_anteriores,N_telephony_provider_permission_denied_anteriores,N_dial_busy_anteriores,N_telephony_provider_unavailable_anteriores,N_agent_hangup_anteriores,N_error_asr_anteriores,N_error_retell_anteriores,N_dial_failed_anteriores,N_max_duration_reached_anteriores,N_ivr_reached_anteriores
count,45676.000000,45675.000000,45675.000000,2003.000000,33792.000000,0.0,0.0,45676.000000,45676.000000,45667.000000,...,45676.000000,45676.000000,45676.000000,45676.000000,45676.000000,45676.000000,45676.000000,45676.000000,45676.0,45676.000000
mean,81597.768697,4.580142,0.589970,80.777534,2430.415927,NaN,NaN,1798.069336,0.034022,102.010204,...,1.128689,0.166740,40.651546,27.381754,0.038094,0.000022,0.025046,0.000547,0.0,0.007794
std,44338.790169,1.077841,3.555701,554.676728,12719.707541,NaN,NaN,10992.371417,0.181288,292.846188,...,6.297829,1.053948,195.540432,87.030976,0.216036,0.004679,0.156266,0.023389,0.0,0.209633
min,5630.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
25%,39956.500000,4.000000,0.000000,1.600000,0.000000,NaN,NaN,0.000000,0.000000,2.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
50%,80306.500000,4.000000,0.000000,4.300000,0.000000,NaN,NaN,0.000000,0.000000,6.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
75%,122233.500000,5.000000,0.000000,4.800000,0.000000,NaN,NaN,0.000000,0.000000,11.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
max,156186.000000,11.000000,244.400000,6094.000000,665266.000000,NaN,NaN,665266.000000,1.000000,1963.000000,...,84.000000,17.000000,1303.000000,588.000000,8.000000,1.000000,1.000000,1.000000,0.0,11.000000


In [14]:
df_original.columns

Index(['id', 'created_at', 'Nome', 'Email', 'data', 'Numero', 'status',
       'call_id', 'call_type', 'agent_id', 'agent_version', 'agent_name',
       'transcript', 'recording_url', 'disconnection_reason',
       'eleven_labs_cost', 'LLM', 'LLM_cost', 'combined_cost', 'call_summary',
       'LLM_token_usage', 'from_number', 'to_number', 'Duracao', 'Marcada',
       'segmento', 'equipe', 'Duracao_num', 'target',
       'n_tentativas_anteriores', 'horas_desde_primeiro_contato', 'ddd',
       'Regiao', 'n_voicemail_reached_anteriores',
       'n_dial_no_answer_anteriores', 'n_inactivity_anteriores',
       'N_invalid_destination_anteriores', 'N_user_hangup_anteriores',
       'N_user_declined_anteriores',
       'N_telephony_provider_permission_denied_anteriores',
       'N_dial_busy_anteriores', 'N_telephony_provider_unavailable_anteriores',
       'N_agent_hangup_anteriores', 'N_error_asr_anteriores',
       'N_error_retell_anteriores', 'N_dial_failed_anteriores',
       'N_max_durati

In [15]:
# 1. Definir a lista de colunas selecionadas para o Lead Scoring
colunas_ml = [
    # Target
    'target',
    # Informações Geográficas
    'ddd', 'Regiao',
    # Métricas de Tentativas
    'n_tentativas_anteriores', 'horas_desde_primeiro_contato',
    # Contagem de Razões de Disconexão/Status (Anteriores)
    'n_voicemail_reached_anteriores',
    'n_dial_no_answer_anteriores',
    'n_inactivity_anteriores',
    'N_invalid_destination_anteriores',
    'N_user_hangup_anteriores',
    'N_user_declined_anteriores',
    'N_telephony_provider_permission_denied_anteriores',
    'N_dial_busy_anteriores',
    'N_telephony_provider_unavailable_anteriores',
    'N_agent_hangup_anteriores',
    'N_error_asr_anteriores',
    'N_error_retell_anteriores',
    'N_dial_failed_anteriores',
    'N_max_duration_reached_anteriores',
    'N_ivr_reached_anteriores',
    # Status da Última Interação
    'ultima_disconnection_reason'
]

# 2. Criar o DataFrame final filtrado
df_ml = df_original[colunas_ml].copy()

print(f"DataFrame filtrado com sucesso! Agora temos {df_ml.shape[1]} colunas.")
display(df_ml.head())

DataFrame filtrado com sucesso! Agora temos 21 colunas.


,target,ddd,Regiao,n_tentativas_anteriores,horas_desde_primeiro_contato,n_voicemail_reached_anteriores,n_dial_no_answer_anteriores,n_inactivity_anteriores,N_invalid_destination_anteriores,N_user_hangup_anteriores,...,N_telephony_provider_permission_denied_anteriores,N_dial_busy_anteriores,N_telephony_provider_unavailable_anteriores,N_agent_hangup_anteriores,N_error_asr_anteriores,N_error_retell_anteriores,N_dial_failed_anteriores,N_max_duration_reached_anteriores,N_ivr_reached_anteriores,ultima_disconnection_reason
115958,0,NaN,NaN,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,primeiro_contato
80958,0,NaN,NaN,1.0,0.106000,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,invalid_destination
1669,0,NaN,NaN,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,primeiro_contato
133528,0,NaN,NaN,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,primeiro_contato
1625,0,NaN,NaN,1.0,0.086529,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,invalid_destination


In [19]:
# Calcular a soma de valores nulos por coluna
null_count = df_ml.isnull().sum()

# Calcular a porcentagem de valores nulos
null_percentage = (df_ml.isnull().sum() / len(df_ml)) * 100

# Criar um DataFrame de resumo
resumo_nulos = pd.DataFrame({
    'Valores Nulos': null_count,
    'Porcentagem (%)': null_percentage
})

# Filtrar apenas colunas que possuem nulos e ordenar
resumo_nulos = resumo_nulos[resumo_nulos['Valores Nulos'] > 0].sort_values(by='Valores Nulos', ascending=False)

print(f"Resumo de valores nulos no df_ml (Total de linhas: {len(df_ml)}):")
if not resumo_nulos.empty:
    display(resumo_nulos)
else:
    print("Nenhum valor nulo encontrado!")

Resumo de valores nulos no df_ml (Total de linhas: 45676):
Nenhum valor nulo encontrado!


In [18]:
# 1. Preencher DDD e Regiao com a moda (valor mais frequente)
df_ml['ddd'] = df_ml['ddd'].fillna(df_ml['ddd'].mode()[0])
df_ml['Regiao'] = df_ml['Regiao'].fillna(df_ml['Regiao'].mode()[0])

# 2. Preencher métricas numéricas nulas com 0
colunas_zero = ['n_tentativas_anteriores', 'horas_desde_primeiro_contato']
df_ml[colunas_zero] = df_ml[colunas_zero].fillna(0)

# Verificar se ainda restam nulos no df_ml
print("Valores nulos restantes no df_ml:")
print(df_ml.isnull().sum())

display(df_ml.head())

Valores nulos restantes no df_ml:
target                                               0
ddd                                                  0
Regiao                                               0
n_tentativas_anteriores                              0
horas_desde_primeiro_contato                         0
n_voicemail_reached_anteriores                       0
n_dial_no_answer_anteriores                          0
n_inactivity_anteriores                              0
N_invalid_destination_anteriores                     0
N_user_hangup_anteriores                             0
N_user_declined_anteriores                           0
N_telephony_provider_permission_denied_anteriores    0
N_dial_busy_anteriores                               0
N_telephony_provider_unavailable_anteriores          0
N_agent_hangup_anteriores                            0
N_error_asr_anteriores                               0
N_error_retell_anteriores                            0
N_dial_failed_anteriores       

,target,ddd,Regiao,n_tentativas_anteriores,horas_desde_primeiro_contato,n_voicemail_reached_anteriores,n_dial_no_answer_anteriores,n_inactivity_anteriores,N_invalid_destination_anteriores,N_user_hangup_anteriores,...,N_telephony_provider_permission_denied_anteriores,N_dial_busy_anteriores,N_telephony_provider_unavailable_anteriores,N_agent_hangup_anteriores,N_error_asr_anteriores,N_error_retell_anteriores,N_dial_failed_anteriores,N_max_duration_reached_anteriores,N_ivr_reached_anteriores,ultima_disconnection_reason
115958,0,11,SP,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,primeiro_contato
80958,0,11,SP,1.0,0.106000,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,invalid_destination
1669,0,11,SP,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,primeiro_contato
133528,0,11,SP,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,primeiro_contato
1625,0,11,SP,1.0,0.086529,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,invalid_destination


In [20]:
from google.colab import files

# Salvar o DataFrame como CSV
file_name = 'df_ml_LS.csv'
df_ml.to_csv(file_name, index=False)

# Baixar o arquivo
files.download(file_name)

print(f"Arquivo '{file_name}' gerado e download iniciado.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Arquivo 'df_ml_LS.csv' gerado e download iniciado.
